# 01 - Pipeline ETL

Notebook do Projeto de Integração sobre satisfação do passageiro em aeroportos.

A ideia aqui é deixar o processo bem dividido, com comentários e saídas fáceis de conferir.

## 1. Preparação do ambiente

Execute esta célula para instalar as bibliotecas necessárias no Google Colab.

In [ ]:
# Instala as dependências do projeto.
# No Colab, normalmente pandas já vem instalado, mas deixo explícito para evitar erro.
!pip install -q pandas openpyxl matplotlib

## 2. Definição da pasta do projeto

No Colab, coloque este notebook dentro da raiz do repositório ou ajuste a variável `PROJECT_DIR`.

In [ ]:
from pathlib import Path
import os

# Ajuste este caminho se o repositório estiver em outro local no Colab.
PROJECT_DIR = Path.cwd()

# Se o notebook estiver dentro da pasta notebooks, volto um nível para chegar à raiz.
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

os.chdir(PROJECT_DIR)
print('Pasta atual do projeto:', PROJECT_DIR)

## 3. Execução do pipeline

In [ ]:
# Pipeline ETL:
# 1) Extrai os dados das três planilhas XLSX.
# 2) Transforma os dados em Python/Pandas.
# 3) Carrega o resultado em CSV e SQLite.
!python src/etl/run_etl.py

## 4. Conferência das tabelas geradas

A célula abaixo mostra quantas linhas existem em cada tabela do Data Warehouse.

In [ ]:
import sqlite3
import pandas as pd

db_path = Path('data/warehouse/aeroportos_dw_etl.sqlite')
print('Banco usado:', db_path)

with sqlite3.connect(db_path) as conn:
    tabelas = pd.read_sql_query("""
        SELECT name AS tabela
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
    """, conn)

    resumo = []
    for tabela in tabelas['tabela']:
        total = conn.execute(f'SELECT COUNT(*) FROM {tabela}').fetchone()[0]
        resumo.append({'tabela': tabela, 'linhas': total})

pd.DataFrame(resumo)

## 5. Exemplo de consulta analítica

Aqui já faço uma consulta simples para validar o resultado final: média de satisfação por ano.

In [ ]:
with sqlite3.connect(Path('data/warehouse/aeroportos_dw_etl.sqlite')) as conn:
    consulta = pd.read_sql_query("""
        SELECT
            dt.ano,
            COUNT(*) AS total_entrevistas,
            ROUND(AVG(fe.satisfacao_geral), 3) AS media_satisfacao_geral,
            ROUND(100.0 * AVG(CASE WHEN fe.satisfacao_geral >= 4 THEN 1 ELSE 0 END), 2) AS percentual_positivo
        FROM fato_entrevista fe
        JOIN dim_tempo dt ON dt.data_key = fe.data_key
        GROUP BY dt.ano
        ORDER BY dt.ano
    """, conn)

consulta